In [0]:
import oracledb
import pandas as pd
import sys

sys.path.append("/Volumes/opsanalytics_adb_workspace01/default/oracle_connections")
from db_config import get_connection 

In [0]:
lab_mapping_tables_list = ['LAB_KPI_AP_PATIENT_SETTING', 
                           'LAB_KPI_AP_TAT_TARGETS', 
                           'LAB_KPI_DEFINITIONS', 
                           'LAB_KPI_GI_CODES', 
                           'LAB_KPI_SCC_CLINICTYPE',
                           'LAB_KPI_SCC_ICU',
                           'LAB_KPI_SCC_TEST_CODES',
                           'LAB_KPI_SITE_NAMES',
                           'LAB_KPI_SUN_ICU',
                           'LAB_KPI_SUN_LOCTYPE',
                           'LAB_KPI_SUN_TEST_CODES',
                           'LAB_KPI_TURNAROUND_TARGETS']
lab_data_tables_list = ['OAO_PRODUCTION.LAB_KPI_POWERPATH_CYTO_BIOPSY', 
                        'OAO_PRODUCTION.LAB_KPI_PREPROCESSED_DAILY']

In [0]:
# 3. Oracle applies the filter before returning data to Databricks.
oracle_jdbc_url = dbutils.secrets.get(
    scope="oao_secrets",
    key="ORACLE_JDBC_URL",
)

oracle_password = dbutils.secrets.get(
    scope="oao_secrets",
    key="OAO_PRODUCTION",
)

def oracle_reader(dbtable):
    return (
        spark.read.format("jdbc")
        .option("url", oracle_jdbc_url)
        .option("dbtable", dbtable)
        .option("user", "OAO_PRODUCTION")
        .option("password", oracle_password)
        .option("driver", "oracle.jdbc.OracleDriver")
        .option("oracle.net.ssl_server_dn_match", "true")
        .option("fetchsize", "10000")
    )

In [0]:
for tbl in lab_mapping_tables_list:
    df = oracle_reader(f"OAO_PRODUCTION.{tbl}").load()
    (
        df.write
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"opsanalytics_adb_workspace01.lab.{tbl.lower()}")
    )
    print(f"{tbl}: {df.count()} rows written")

In [0]:
num_partitions = 8

for full_name in lab_data_tables_list:
    tbl = full_name.split(".")[-1]          # strip OAO_PRODUCTION.
    # wrap the table in a subquery that adds a hash bucket column
    subquery = f"(SELECT t.*, ORA_HASH(ROWID, {num_partitions - 1}) AS part_id FROM {full_name} t)"
    df = (
        oracle_reader(subquery)
        .option("partitionColumn", "part_id")
        .option("lowerBound", "0")
        .option("upperBound", str(num_partitions))
        .option("numPartitions", str(num_partitions))
        .load()
        .drop("part_id")
    )
    target = f"opsanalytics_adb_workspace01.lab_staging.{tbl.lower()}"
    (
        df.write
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(target)
    )
    print(f"{full_name} -> {target}")